# Clairvue — Management Claim & Peer Evidence Assistant

Clairvue is a RAG system that validates bank management statements (e.g. earnings-call claims like *"Consumer credit remains resilient"*) against formal SEC disclosures, quantitative XBRL metrics, and peer-bank evidence. Given a claim, it decomposes the claim into atomic, independently verifiable assertions, retrieves grounded evidence for each one, and returns a structured assessment — **supported / partially supported / contradicted / insufficient evidence** — with citations back to the actual filings and metrics. It also supports multi-bank peer comparison on a given risk theme.

This notebook was built as a take-home demo for a **Mistral AI interview challenge**. All embeddings and generation run exclusively on Mistral models (`mistral-embed` + a chat model), with no other model provider anywhere in the pipeline.

**Scope:** Prototype uses public SEC filings from JPMorgan Chase, Bank of America, and Citigroup, Q4 2022 through Q4 2023. The risk theme covered is consumer credit quality (charge-offs, delinquencies, provisions, "normalization" language).

Run the cells below in order. The first cell clones the repo and installs dependencies; the data (chunked filings, embeddings, ChromaDB index, and curated metrics) is expected to already exist in Google Drive at `MyDrive/clairvue_data` (built ahead of time via `scripts/download_filings.py` + `scripts/build_index.py`).

In [ ]:
# Install dependencies
import subprocess, sys

!git clone https://github.com/dannyyuanxu/clairvue.git 2>/dev/null || echo "already cloned"
%cd clairvue

# Run pip quietly and only surface output if it actually fails. The "ERROR: pip's
# dependency resolver..." lines you may have seen are NOT failures -- they're pre-existing
# conflicts among Colab's own pre-installed packages (google-adk / opentelemetry-exporter-*),
# none of which Clairvue uses. The install itself succeeds, so we hide that noise here.
print("Installing dependencies (~1 min)...")
_pip = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    capture_output=True, text=True,
)
if _pip.returncode != 0:
    print(_pip.stdout)
    print(_pip.stderr)
    raise RuntimeError("pip install failed -- see output above")
print("Dependencies installed.")

# Mount Google Drive (data lives here). A brand-new Colab runtime always prompts for auth
# once (the VM is ephemeral and doesn't persist the token) -- to avoid re-auth, use
# Runtime > "Restart session" rather than "Disconnect and delete runtime". force_remount=False
# makes re-running this cell within the same session a no-op (no second popup).
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Load API key from Colab Secrets (Settings -> Secrets -> add MISTRAL_API_KEY)
import os
from google.colab import userdata

os.environ["MISTRAL_API_KEY"] = userdata.get("MISTRAL_API_KEY")
os.environ["EMBEDDING_MODEL"] = "mistral-embed"
os.environ["LLM_MODEL"] = "mistral-medium-3-5"           # Medium for demo quality
os.environ["ENRICHMENT_MODEL"] = "mistral-small-latest"  # Small always for enrichment

# Point to Drive data
DATA_DIR = "/content/drive/MyDrive/clairvue_data"
os.environ["CHROMA_PERSIST_DIR"] = f"{DATA_DIR}/chroma_db"

print("Setup complete.")

In [ ]:
print('Checking for latest changes from the repository...')
!git pull -q
print('Repository updated.')

In [ ]:
import sys
sys.path.insert(0, '/content/clairvue')

from src.vector_store import load_collection
from src.retrieval import EvidenceRetriever

collection = load_collection()
print(f"Loaded ChromaDB collection: {collection.count()} chunks indexed")

## Demo 1 — Claim Validation

Test whether a management statement is supported by formal filing evidence.

In [ ]:
from src.generation import answer_claim
from src.demo import display_claim_assessment

statement = "Consumer credit remains resilient, and losses are normalizing in line with expectations."
result = answer_claim(statement, ticker="JPM", verbose=False)
display_claim_assessment(result, verbose=False)
# To see analyst follow-up questions: set verbose=True in both calls

## Demo 2 — Peer Comparison

Compare consumer-credit risk evidence across JPMorgan, Bank of America, and Citigroup.

In [ ]:
from src.generation import compare_peers
from src.demo import display_peer_comparison

question = "Which of JPMorgan, Bank of America, and Citigroup shows the strongest evidence of increasing consumer-credit pressure in 2023?"
result = compare_peers(question, risk_theme="consumer_credit")
display_peer_comparison(result)

## Demo 3 — Appropriate Abstention

Test that the system correctly abstains when evidence is insufficient to support a definitive claim.

In [ ]:
from src.demo import display_claim_assessment

result3 = answer_claim(
    # "Does the evidence prove that consumer credit losses have peaked and will improve from here?",
    "Consumer credit deterioration in 2023 was concentrated among younger, first-time borrowers.",
    risk_theme="consumer_credit",
)
display_claim_assessment(result3)

## Demo 4 — Real CEO statement (Jamie Dimon, 2023-01-13 earnings)

A real macro statement from JPMorgan's CEO, decomposed into atomic claims and assessed claim-by-claim against the filing evidence and metrics.

In [ ]:
from src.demo import display_claim_assessment

statement = "The U.S. economy currently remains strong with consumers still spending excess cash and businesses healthy. However, we still do not know the ultimate effect of the headwinds coming from geopolitical tensions including the war in Ukraine, the vulnerable state of energy and food supplies, persistent inflation that is eroding purchasing power and has pushed interest rates higher, and the unprecedented quantitative tightening."
result4 = answer_claim(statement, risk_theme="consumer_credit")
display_claim_assessment(result4)

## Demo 5 — Real CEO statement (Jamie Dimon, FY2023 results, 2024-01-12)

Jamie Dimon's earnings commentary on JPMorgan's record 2023, explicitly flagging "over-earning" on net interest income and credit and expecting both to normalize. Assessed against JPM's own filings and metrics (note `ticker="JPM"`).

To show exactly what `verbose` adds, we run the assessment **once** (`verbose=True`) and render the *same result* two ways — first hiding the analyst follow-up questions, then showing them. Rendering one result twice (rather than calling `answer_claim` twice) keeps everything else identical, so the only difference between the two views is the follow-up questions. Calling `answer_claim` twice would re-decompose and re-assess from scratch, producing two independent generations whose wording differs run-to-run.

In [ ]:
from src.generation import answer_claim
from src.demo import display_claim_assessment

dimon_2023 = (
    "Our record results in 2023 reflect over-earning on both NII and credit, but we remain "
    "confident in our ability to continue to deliver very healthy returns even after they normalize"
    # "normalize. The U.S. economy continues to be resilient, with consumers still spending, "
    "and markets currently expect a soft landing."
)

# Without verbose: the core verdict, rationale, metrics, and evidence per claim.
result5 = answer_claim(dimon_2023, ticker="JPM", verbose=True)
display_claim_assessment(result5, verbose=False)

# View 2 -- the SAME result5, now showing the analyst follow-up questions.
# Only difference from View 1 above is the follow-up questions; everything else is identical.
# display_claim_assessment(result5, verbose=True)

In [ ]:
# check back on past management statements

# BAC	Bank of America Corporation	10/17/23	Brian Moynihan	AP news article	on slowing spending in credit card
statement = (
"[in late 2023] We did this in a healthy but slowing economy that saw US consumer spending still ahead of last year but continuing to slow."
)
# Without verbose: the core verdict, rationale, metrics, and evidence per claim.
result6 = answer_claim(statement, ticker="BAC", verbose=True)
display_claim_assessment(result6, verbose=False)

## Architecture Notes

**Pipeline:** SEC filing HTML is parsed with `sec-parser` into clean section-aware text, split into section-aware chunks tagged with risk theme and metadata, embedded with `mistral-embed`, and indexed into ChromaDB with metadata filters (ticker, risk theme, filing type, period). At query time, a claim is decomposed into atomic assertions; each assertion is retrieved independently (plus a contradiction-query expansion pass to deliberately surface counter-evidence), formatted into a grounded prompt alongside curated XBRL metrics, and assessed by a Mistral chat model constrained to return structured JSON citing only the supplied evidence. Peer comparison follows the same retrieval-then-structured-generation pattern across multiple banks in a single grounded call.

**Why this design:** Pure Python (no LangChain) so every retrieval and assessment step is individually inspectable and explainable line-by-line — important for a live walkthrough. All model calls go through one `LLMClient` class, so swapping providers or models is a one-file change. Two-stage caching (parse once, re-chunk/re-embed independently) means re-tuning retrieval doesn't require re-downloading filings, and a model swap doesn't require re-parsing.

**Production extension path:** the same architecture (per-claim isolated retrieval, structured-JSON generation, explicit citation requirements, explicit abstention on insufficient evidence) generalizes beyond this 3-bank/5-quarter prototype to a private-deployment research tool covering a broader filer universe and risk themes, with the model abstraction layer making it straightforward to swap in a fine-tuned or self-hosted model behind the same `LLMClient` interface.